# Incident Response Runbook: Exfiltration - Exfiltration Over C2 Channel

**Tactic:** Exfiltration
**Technique:** Exfiltration Over C2 Channel (T1041)
**Severity:** CRITICAL

## Overview

This runbook provides step-by-step incident response procedures for Exfiltration Over C2 Channel activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

# Add path for custom integrations
sys.path.append(os.path.join(os.path.dirname(__file__), '..', '..'))

# Import security integrations
from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_data_collector import CrowdStrikeDataCollector
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Initialize integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeDataCollector()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

detection_time = datetime.now().isoformat()
detection_actions = []
affected_systems = []
threat_indicators = []

# 1. Analyze data exfiltration over C2 channels in Splunk
print(f"\n[DETECTION] Analyzing data exfiltration over C2 channels...")
try:
    # Query Splunk for suspicious data transfers over C2 protocols
    splunk_query = """
    index=* (sourcetype=network OR sourcetype=firewall OR sourcetype=proxy)
    | search protocol IN ("http", "https", "dns", "smtp", "ftp", "ssh", "tcp", "udp")
    | eval data_transfer_mb = bytes_out / 1024 / 1024
    | eval risk_score = case(
        protocol=="dns" AND data_transfer_mb>10 AND query_count>100, 9,
        protocol=="http" AND data_transfer_mb>50 AND status_code IN (200, 201), 8,
        protocol=="https" AND data_transfer_mb>100 AND connection_count>10, 8,
        protocol=="smtp" AND data_transfer_mb>20 AND recipient_count>5, 7,
        protocol=="ftp" AND data_transfer_mb>200, 8,
        protocol=="ssh" AND data_transfer_mb>50 AND connection_count>3, 7,
        protocol IN ("tcp", "udp") AND data_transfer_mb>100 AND dest_port NOT IN (80, 443, 53, 25, 22), 8,
        1==1, 3
    )
    | where risk_score >= 6 AND data_transfer_mb > 10
    | stats sum(data_transfer_mb) as total_mb, count by src_ip, dest_ip, protocol, dest_port
    | sort -total_mb
    """

    alert_results = splunk.execute_query(splunk_query)
    if alert_results and len(alert_results) > 0:
        print(f"   Found {len(alert_results)} suspicious data exfiltration attempts")

        for alert in alert_results[:10]:  # Top 10 exfiltration alerts
            affected_systems.append({
                'ip': alert.get('src_ip'),
                'protocol': alert.get('protocol'),
                'dest_ip': alert.get('dest_ip'),
                'dest_port': alert.get('dest_port'),
                'data_exfiltrated_mb': alert.get('total_mb', 0),
                'connection_count': alert.get('count', 0)
            })

        detection_actions.append({
            'action': 'splunk_exfiltration_analysis',
            'findings': f"{len(alert_results)} suspicious exfiltration attempts detected",
            'method': 'Splunk',
            'status': 'success',
            'timestamp': detection_time
        })
    else:
        print(f"   No suspicious data exfiltration detected in recent logs")

except Exception as e:
    print(f"   Error analyzing exfiltration: {e}")

# 2. Check CrowdStrike for data exfiltration indicators
print(f"\n[DETECTION] Checking CrowdStrike for exfiltration indicators...")
try:
    # Look for processes performing large data transfers
    cs_indicators = crowdstrike.search_indicators({
        'indicator_type': 'data_exfiltration',
        'time_range': '24h'
    })

    if cs_indicators:
        for indicator in cs_indicators:
            if indicator.get('data_volume_mb', 0) > 50:
                threat_indicators.append({
                    'type': 'data_exfiltration',
                    'value': indicator.get('connection_string'),
                    'data_volume_mb': indicator.get('data_volume_mb'),
                    'source': 'CrowdStrike',
                    'risk_score': indicator.get('risk_score')
                })

        detection_actions.append({
            'action': 'crowdstrike_exfiltration_check',
            'findings': f"{len(threat_indicators)} high-volume data transfers detected",
            'method': 'CrowdStrike',
            'status': 'success',
            'timestamp': detection_time
        })
        print(f"   Found {len(threat_indicators)} high-volume data exfiltration indicators")
    else:
        print(f"   No high-volume data transfers found in CrowdStrike")

except Exception as e:
    print(f"   Error checking CrowdStrike: {e}")

# 3. Analyze unusual outbound data patterns
print(f"\n[DETECTION] Analyzing unusual outbound data patterns...")
try:
    # Look for anomalous outbound traffic patterns
    outbound_query = """
    index=* sourcetype=network
    | eval data_mb = bytes_out / 1024 / 1024
    | where data_mb > 20
    | stats sum(data_mb) as total_outbound_mb, count by src_ip, dest_ip
    | eval avg_mb_per_connection = total_outbound_mb / count
    | where avg_mb_per_connection > 10
    | sort -total_outbound_mb
    """

    outbound_results = splunk.execute_query(outbound_query)
    if outbound_results:
        for result in outbound_results[:5]:  # Top 5 outbound patterns
            threat_indicators.append({
                'type': 'unusual_outbound',
                'value': f"{result.get('src_ip')} -> {result.get('dest_ip')}",
                'total_data_mb': result.get('total_outbound_mb'),
                'connections': result.get('count'),
                'source': 'Splunk outbound analysis',
                'risk_score': 7
            })

        detection_actions.append({
            'action': 'outbound_pattern_analysis',
            'findings': f"{len(outbound_results)} unusual outbound patterns detected",
            'method': 'Splunk',
            'status': 'success',
            'timestamp': detection_time
        })
        print(f"   Found {len(outbound_results)} unusual outbound data patterns")

except Exception as e:
    print(f"   Error analyzing outbound patterns: {e}")

# 4. Enrich threat intelligence via MISP
print(f"\n[DETECTION] Enriching threat intelligence...")
try:
    for indicator in threat_indicators:
        misp_results = misp.search_indicators(indicator['value'])
        if misp_results:
            indicator['misp_enrichment'] = misp_results
            print(f"   Enriched indicator: {indicator['value'][:50]}...")

    detection_actions.append({
        'action': 'threat_intel_enrichment',
        'findings': f"Enriched {len([i for i in threat_indicators if 'misp_enrichment' in i])} indicators",
        'method': 'MISP',
        'status': 'success',
        'timestamp': detection_time
    })

except Exception as e:
    print(f"   Error enriching threat intelligence: {e}")

# 5. Create IRIS incident case
print(f"\n[DETECTION] Creating IRIS incident case...")
try:
    total_data_exfiltrated = sum([s.get('data_exfiltrated_mb', 0) for s in affected_systems])
    incident_data = {
        'title': 'Data Exfiltration Over C2 Channel Detected',
        'description': f'Detected suspicious data exfiltration totaling {total_data_exfiltrated:.1f} MB over C2 channels with {len(threat_indicators)} indicators',
        'severity': 'CRITICAL',
        'tactic': 'Exfiltration',
        'technique': 'Exfiltration Over C2 Channel (T1041)',
        'affected_systems': affected_systems,
        'threat_indicators': threat_indicators,
        'detection_time': detection_time
    }

    incident_id = iris.create_incident(incident_data)
    if incident_id:
        detection_actions.append({
            'action': 'incident_creation',
            'findings': f"Incident {incident_id} created",
            'method': 'IRIS',
            'status': 'success',
            'timestamp': detection_time
        })
        print(f"   IRIS incident case created: {incident_id}")
    else:
        print(f"   Failed to create IRIS incident case")

except Exception as e:
    print(f"   Error creating IRIS case: {e}")

print(f"\n Detection complete:")
print(f"  - Affected systems identified: {len(affected_systems)}")
print(f"  - Threat indicators found: {len(threat_indicators)}")
print(f"  - Total data potentially exfiltrated: {total_data_exfiltrated:.1f} MB")
print(f"  - IRIS case created: {incident_id if 'incident_id' in locals() else 'N/A'}")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

containment_time = datetime.now().isoformat()
containment_actions = []
isolated_hosts = []
blocked_destinations = []
disabled_accounts = []

# 1. Isolate affected systems
print(f"\n[CONTAINMENT] Isolating affected systems...")
try:
    for system in affected_systems:
        if system.get('ip'):
            # Use CrowdStrike to isolate host
            isolation_result = crowdstrike.isolate_host_by_ip(system['ip'])
            if isolation_result and isolation_result.get('status') == 'isolated':
                isolated_hosts.append(system['ip'])
                containment_actions.append({
                    'action': 'host_isolation',
                    'target': system['ip'],
                    'method': 'CrowdStrike',
                    'status': 'success',
                    'timestamp': containment_time
                })
                print(f"   Isolated host: {system['ip']}")
            else:
                print(f"   Failed to isolate host: {system['ip']}")
        else:
            # Use Shuffle for network isolation
            network_isolation = shuffle.isolate_system(system.get('hostname', system['ip']))
            if network_isolation:
                isolated_hosts.append(system.get('hostname', system['ip']))
                containment_actions.append({
                    'action': 'network_isolation',
                    'target': system.get('hostname', system['ip']),
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': containment_time
                })
                print(f"   Network isolated: {system.get('hostname', system['ip'])}")
except Exception as e:
    print(f"   Error isolating systems: {e}")

# 2. Block exfiltration destinations
print(f"\n[CONTAINMENT] Blocking exfiltration destinations...")
try:
    for indicator in threat_indicators:
        if indicator.get('type') in ['data_exfiltration', 'unusual_outbound']:
            # Extract destination IP from indicator
            if '->' in indicator['value']:
                dest_ip = indicator['value'].split('->')[1].strip()
            else:
                dest_ip = indicator.get('value', '').split(':')[0]

            if dest_ip:
                # Use Shuffle to block destination
                block_result = shuffle.block_ip_domain(dest_ip)
                if block_result:
                    blocked_destinations.append(dest_ip)
                    containment_actions.append({
                        'action': 'destination_block',
                        'target': dest_ip,
                        'method': 'Shuffle',
                        'status': 'success',
                        'timestamp': containment_time
                    })
                    print(f"   Blocked exfiltration destination: {dest_ip}")
except Exception as e:
    print(f"   Error blocking destinations: {e}")

# 3. Disable compromised accounts
print(f"\n[CONTAINMENT] Disabling compromised accounts...")
try:
    # Identify accounts associated with exfiltration activity
    suspicious_accounts = []
    for system in affected_systems:
        if system.get('ip'):
            # Query Splunk for user activity on affected systems
            user_query = f"""
            index=* src_ip="{system['ip']}"
            | search protocol IN ("http", "https", "ftp", "ssh")
            | eval data_mb = bytes_out / 1024 / 1024
            | where data_mb > 5
            | stats sum(data_mb) as total_mb by user
            | where total_mb > 20
            | sort -total_mb
            """
            user_results = splunk.execute_query(user_query)
            if user_results:
                suspicious_accounts.extend([u.get('user') for u in user_results])

    # Remove duplicates
    suspicious_accounts = list(set(suspicious_accounts))

    for account in suspicious_accounts[:5]:  # Limit to top 5 suspicious accounts
        # Use Shuffle to disable account
        disable_result = shuffle.disable_account(account)
        if disable_result:
            disabled_accounts.append(account)
            containment_actions.append({
                'action': 'account_disable',
                'target': account,
                'method': 'Shuffle',
                'status': 'success',
                'timestamp': containment_time
            })
            print(f"   Disabled account: {account}")
except Exception as e:
    print(f"   Error disabling accounts: {e}")

# 4. Enable enhanced monitoring for data exfiltration
print(f"\n[CONTAINMENT] Enabling enhanced monitoring...")
try:
    # Create Splunk correlation search for data exfiltration over C2
    correlation_search = {
        'name': f'C2_Exfiltration_Monitoring_{incident_id}',
        'search': """
        index=* (sourcetype=network OR sourcetype=firewall)
        | search protocol IN ("http", "https", "dns", "smtp", "ftp", "ssh", "tcp", "udp")
        | eval data_mb = bytes_out / 1024 / 1024
        | eval risk_score = case(
            protocol=="dns" AND data_mb>5 AND query_count>50, 8,
            protocol=="http" AND data_mb>25, 7,
            protocol=="https" AND data_mb>50 AND connection_count>5, 8,
            protocol=="ftp" AND data_mb>100, 8,
            protocol=="ssh" AND data_mb>25, 7,
            protocol IN ("tcp", "udp") AND data_mb>50 AND dest_port NOT IN (80, 443, 53), 7,
            1==1, 3
        )
        | where risk_score >= 6 AND data_mb > 5
        | stats sum(data_mb) as total_mb by src_ip, dest_ip, protocol
        """,
        'alert_condition': 'count > 0',
        'alert_threshold': 1
    }

    monitoring_result = splunk.create_correlation_search(correlation_search)
    if monitoring_result:
        containment_actions.append({
            'action': 'enhanced_monitoring',
            'target': f"Correlation search {correlation_search['name']}",
            'method': 'Splunk',
            'status': 'success',
            'timestamp': containment_time
        })
        print(f"   Enhanced monitoring enabled in Splunk")

    # Enable CrowdStrike enhanced detection for exfiltration
    cs_monitoring = crowdstrike.enable_enhanced_detection('data_exfiltration_c2')
    if cs_monitoring:
        containment_actions.append({
            'action': 'cs_enhanced_detection',
            'target': 'CrowdStrike EDR',
            'method': 'CrowdStrike',
            'status': 'success',
            'timestamp': containment_time
        })
        print(f"   Enhanced detection enabled in CrowdStrike")

except Exception as e:
    print(f"   Error enabling enhanced monitoring: {e}")

print(f"\n Containment complete:")
print(f"  - Hosts isolated: {len(isolated_hosts)}")
print(f"  - Destinations blocked: {len(blocked_destinations)}")
print(f"  - Accounts disabled: {len(disabled_accounts)}")
print(f"  - Enhanced monitoring: ")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

eradication_time = datetime.now().isoformat()
eradication_actions = []
terminated_processes = []
deleted_files = []
reset_credentials = []

# 1. Terminate processes performing exfiltration
print(f"\n[ERADICATION] Terminating exfiltration processes...")
try:
    for system in affected_systems:
        if system.get('ip'):
            # Get processes associated with large data transfers
            exfiltration_processes = crowdstrike.get_processes_by_data_transfer(system['ip'], min_mb=10)
            if exfiltration_processes:
                for process in exfiltration_processes:
                    if process.get('bytes_out', 0) > 10 * 1024 * 1024:  # > 10MB
                        # Terminate process
                        terminate_result = crowdstrike.terminate_process(process['process_id'])
                        if terminate_result:
                            terminated_processes.append({
                                'process': process.get('process_name'),
                                'pid': process['process_id'],
                                'host': system['ip'],
                                'data_transferred_mb': process.get('bytes_out', 0) / 1024 / 1024
                            })
                            eradication_actions.append({
                                'action': 'process_termination',
                                'target': f"{process.get('process_name')} (PID: {process['process_id']}) on {system['ip']}",
                                'method': 'CrowdStrike',
                                'status': 'success',
                                'timestamp': eradication_time
                            })
                            print(f"   Terminated exfiltration process: {process.get('process_name')} on {system['ip']}")
except Exception as e:
    print(f"   Error terminating processes: {e}")

# 2. Remove exfiltration tools and malware
print(f"\n[ERADICATION] Removing exfiltration tools and malware...")
try:
    for system in affected_systems:
        if system.get('ip'):
            # Scan for and remove exfiltration-related malware
            malware_scan = crowdstrike.scan_and_remove_malware(system['ip'], 'exfiltration_tools')
            if malware_scan and malware_scan.get('removed_files'):
                for file_path in malware_scan['removed_files']:
                    deleted_files.append({
                        'file': file_path,
                        'host': system['ip']
                    })
                    eradication_actions.append({
                        'action': 'file_removal',
                        'target': f"{file_path} on {system['ip']}",
                        'method': 'CrowdStrike',
                        'status': 'success',
                        'timestamp': eradication_time
                    })
                    print(f"   Removed exfiltration tool: {file_path} on {system['ip']}")

    # Use Shuffle for additional cleanup of exfiltration artifacts
    for indicator in threat_indicators:
        if indicator.get('type') == 'file' and any(term in indicator['value'].lower() for term in ['upload', 'transfer', 'exfil']):
            cleanup_result = shuffle.remove_malicious_file(indicator['value'])
            if cleanup_result:
                deleted_files.append({
                    'file': indicator['value'],
                    'host': 'multiple'
                })
                print(f"   Cleaned up exfiltration file: {indicator['value']}")
except Exception as e:
    print(f"   Error removing exfiltration tools: {e}")

# 3. Reset compromised credentials
print(f"\n[ERADICATION] Resetting compromised credentials...")
try:
    for account in disabled_accounts:
        # Generate new password and reset account
        reset_result = shuffle.reset_account_password(account)
        if reset_result:
            reset_credentials.append(account)
            eradication_actions.append({
                'action': 'credential_reset',
                'target': account,
                'method': 'Shuffle',
                'status': 'success',
                'timestamp': eradication_time
            })
            print(f"   Reset credentials for: {account}")
except Exception as e:
    print(f"   Error resetting credentials: {e}")

# 4. Clear any cached data or temporary files
print(f"\n[ERADICATION] Clearing cached data and temporary files...")
try:
    cleared_items = []
    for system in affected_systems:
        if system.get('ip'):
            # Clear browser caches, temp files, and other potential exfiltration staging areas
            cache_clear = shuffle.clear_system_cache(system['ip'])
            if cache_clear and cache_clear.get('cleared_mb', 0) > 0:
                cleared_items.append({
                    'system': system['ip'],
                    'cleared_mb': cache_clear.get('cleared_mb')
                })
                eradication_actions.append({
                    'action': 'cache_clearing',
                    'target': f"{system['ip']} ({cache_clear.get('cleared_mb')} MB cleared)",
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': eradication_time
                })
                print(f"   Cleared {cache_clear.get('cleared_mb')} MB of cached data on {system['ip']}")

    if cleared_items:
        total_cleared = sum([item['cleared_mb'] for item in cleared_items])
        print(f"   Total cached data cleared: {total_cleared} MB across {len(cleared_items)} systems")

except Exception as e:
    print(f"   Error clearing cached data: {e}")

# 5. Patch exfiltration-related vulnerabilities
print(f"\n[ERADICATION] Patching exfiltration-related vulnerabilities...")
try:
    # Identify common vulnerabilities exploited in data exfiltration
    vulnerabilities = [
        'CVE-2021-44228',  # Log4j
        'CVE-2021-34527',  # PrintNightmare
        'CVE-2021-4034',   # Polkit
        'CVE-2022-22965',  # Spring4Shell
        'CVE-2022-40684',  # Fortinet SSL VPN
    ]

    patched_systems = []
    for system in affected_systems:
        if system.get('ip'):
            # Check and patch vulnerabilities
            patch_result = shuffle.patch_system_vulnerabilities(system['ip'], vulnerabilities)
            if patch_result and patch_result.get('patched_count', 0) > 0:
                patched_systems.append(system['ip'])
                eradication_actions.append({
                    'action': 'vulnerability_patch',
                    'target': f"{system['ip']} ({patch_result.get('patched_count')} patches)",
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': eradication_time
                })
                print(f"   Patched {patch_result.get('patched_count')} vulnerabilities on {system['ip']}")

    if patched_systems:
        print(f"   Vulnerabilities patched on {len(patched_systems)} systems")

except Exception as e:
    print(f"   Error patching vulnerabilities: {e}")

# 6. Verify exfiltration has stopped
print(f"\n[ERADICATION] Verifying exfiltration has stopped...")
try:
    verification_results = []
    for system in affected_systems:
        if system.get('ip'):
            # Check for ongoing exfiltration attempts
            verification_scan = crowdstrike.verify_no_active_exfiltration(system['ip'])
            verification_results.append({
                'system': system['ip'],
                'active_exfiltration': verification_scan.get('active_exfiltration_detected', False),
                'scan_status': verification_scan.get('status', 'unknown')
            })

    clean_systems = [v for v in verification_results if not v['active_exfiltration']]
    eradication_actions.append({
        'action': 'exfiltration_verification',
        'target': f"{len(clean_systems)}/{len(verification_results)} systems with no active exfiltration",
        'method': 'CrowdStrike',
        'status': 'success' if len(clean_systems) == len(verification_results) else 'partial',
        'timestamp': eradication_time
    })

    print(f"   Exfiltration verification complete: {len(clean_systems)}/{len(verification_results)} systems clean")

except Exception as e:
    print(f"   Error verifying exfiltration cessation: {e}")

print(f"\n Eradication complete:")
print(f"  - Processes terminated: {len(terminated_processes)}")
print(f"  - Files removed: {len(deleted_files)}")
print(f"  - Credentials reset: {len(reset_credentials)}")
print(f"  - Cached data cleared: {len(cleared_items) if 'cleared_items' in locals() else 0} systems")
print(f"  - Systems verified clean: {len([v for v in verification_results if not v.get('active_exfiltration', True)])}")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

recovery_time = datetime.now().isoformat()
recovery_actions = []
reenabled_hosts = []
restored_accounts = []

# 1. Re-enable isolated systems
print(f"\n[RECOVERY] Re-enabling isolated systems...")
try:
    for host in isolated_hosts:
        # Use CrowdStrike to re-enable host
        reenable_result = crowdstrike.reenable_host_by_ip(host)
        if reenable_result and reenable_result.get('status') == 'reenabled':
            reenabled_hosts.append(host)
            recovery_actions.append({
                'action': 'host_reenable',
                'target': host,
                'method': 'CrowdStrike',
                'status': 'success',
                'timestamp': recovery_time
            })
            print(f"   Re-enabled host: {host}")
        else:
            # Use Shuffle for network re-enablement
            network_restore = shuffle.restore_system(host)
            if network_restore:
                reenabled_hosts.append(host)
                recovery_actions.append({
                    'action': 'network_restore',
                    'target': host,
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': recovery_time
                })
                print(f"   Restored network access: {host}")
except Exception as e:
    print(f"   Error re-enabling systems: {e}")

# 2. Restore disabled accounts
print(f"\n[RECOVERY] Restoring disabled accounts...")
try:
    for user in disabled_accounts:
        # Restore account access via Shuffle
        restore_result = shuffle.restore_account(user)
        if restore_result:
            restored_accounts.append(user)
            recovery_actions.append({
                'action': 'account_restore',
                'target': user,
                'method': 'Shuffle',
                'status': 'success',
                'timestamp': recovery_time
            })
            print(f"   Restored account: {user}")
except Exception as e:
    print(f"   Error restoring accounts: {e}")

# 3. Validate system functionality
print(f"\n[RECOVERY] Validating system functionality...")
try:
    for system in affected_systems:
        if system.get('ip'):
            # Validate system health after recovery
            health_check = crowdstrike.validate_system_health_by_ip(system['ip'])
            if health_check and health_check.get('status') == 'healthy':
                recovery_actions.append({
                    'action': 'system_validation',
                    'target': system['ip'],
                    'method': 'CrowdStrike',
                    'status': 'success',
                    'timestamp': recovery_time
                })
                print(f"   System health validated: {system['ip']}")
            else:
                print(f"   System health issues detected: {system['ip']}")
except Exception as e:
    print(f"   Error validating system health: {e}")

# 4. Restore monitoring and alerting
print(f"\n[RECOVERY] Restoring monitoring and alerting...")
try:
    # Restore normal Splunk monitoring rules
    normal_monitoring = splunk.restore_normal_monitoring()
    if normal_monitoring:
        recovery_actions.append({
            'action': 'monitoring_restore',
            'target': 'Splunk',
            'method': 'Splunk',
            'status': 'success',
            'timestamp': recovery_time
        })
        print(f"   Restored normal monitoring in Splunk")

    # Restore CrowdStrike normal operations
    cs_normal_ops = crowdstrike.restore_normal_operations()
    if cs_normal_ops:
        recovery_actions.append({
            'action': 'cs_normal_ops',
            'target': 'CrowdStrike',
            'method': 'CrowdStrike',
            'status': 'success',
            'timestamp': recovery_time
        })
        print(f"   Restored normal CrowdStrike operations")

except Exception as e:
    print(f"   Error restoring monitoring: {e}")

# 5. Verify data integrity and assess exfiltration impact
print(f"\n[RECOVERY] Verifying data integrity and assessing exfiltration impact...")
try:
    integrity_checks = []
    total_exfiltrated = 0
    for system in affected_systems:
        if system.get('ip'):
            # Check that no further exfiltration occurred during incident
            integrity_result = shuffle.verify_no_unauthorized_data_transfers(system['ip'])
            integrity_checks.append({
                'system': system['ip'],
                'integrity_ok': integrity_result.get('no_unauthorized_transfers', True),
                'additional_exfiltrated_mb': integrity_result.get('additional_exfiltrated_mb', 0)
            })
            total_exfiltrated += integrity_result.get('additional_exfiltrated_mb', 0)

    systems_with_integrity = [c for c in integrity_checks if c['integrity_ok']]
    recovery_actions.append({
        'action': 'data_integrity_check',
        'target': f"{len(systems_with_integrity)}/{len(integrity_checks)} systems with verified integrity",
        'method': 'Shuffle',
        'status': 'success' if len(systems_with_integrity) == len(integrity_checks) else 'partial',
        'timestamp': recovery_time
    })

    print(f"   Data integrity check complete: {len(systems_with_integrity)}/{len(integrity_checks)} systems OK")
    if total_exfiltrated > 0:
        print(f"   Additional data exfiltrated during incident: {total_exfiltrated:.1f} MB")

except Exception as e:
    print(f"   Error verifying data integrity: {e}")

print(f"\n Recovery complete:")
print(f"  - Hosts re-enabled: {len(reenabled_hosts)}")
print(f"  - Accounts restored: {len(restored_accounts)}")
print(f"  - Systems validated: ")
print(f"  - Data integrity verified: {len([c for c in integrity_checks if c.get('integrity_ok', False)])} systems")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident Actions
print("\n" + "=" * 60)
print("STEP 5: Post-Incident Actions")
print("=" * 60)

post_incident_time = datetime.now().isoformat()
post_incident_actions = []

# 1. Generate comprehensive incident report
print(f"\n[POST-INCIDENT] Generating incident report...")
try:
    total_data_exfiltrated = sum([s.get('data_exfiltrated_mb', 0) for s in affected_systems])
    incident_report = {
        'incident_id': incident_id,
        'technique': 'T1041 - Exfiltration Over C2 Channel',
        'detection_time': detection_time,
        'containment_time': containment_time,
        'eradication_time': eradication_time,
        'recovery_time': recovery_time,
        'post_incident_time': post_incident_time,
        'affected_systems': len(affected_systems),
        'isolated_hosts': len(isolated_hosts),
        'blocked_destinations': len(blocked_destinations),
        'disabled_accounts': len(disabled_accounts),
        'terminated_processes': len(terminated_processes),
        'deleted_files': len(deleted_files),
        'reset_credentials': len(reset_credentials),
        'total_data_exfiltrated_mb': total_data_exfiltrated,
        'reenabled_hosts': len(reenabled_hosts),
        'restored_accounts': len(restored_accounts),
        'threat_indicators': len(threat_indicators),
        'timeline': {
            'detection': detection_time,
            'containment': containment_time,
            'eradication': eradication_time,
            'recovery': recovery_time,
            'closure': post_incident_time
        },
        'actions_taken': {
            'detection': detection_actions,
            'containment': containment_actions,
            'eradication': eradication_actions,
            'recovery': recovery_actions
        },
        'recommendations': [
            'Implement stricter data exfiltration monitoring',
            'Deploy data loss prevention (DLP) solutions',
            'Regular security assessments for data handling procedures',
            'Enhance network traffic analysis and filtering',
            'Conduct user awareness training on data exfiltration risks'
        ]
    }

    # Save report to IRIS
    iris_report = iris.create_incident_report(incident_report)
    if iris_report:
        post_incident_actions.append({
            'action': 'incident_report',
            'target': 'IRIS',
            'method': 'IRIS',
            'status': 'success',
            'timestamp': post_incident_time
        })
        print(f"   Incident report saved to IRIS: {iris_report.get('report_id', 'N/A')}")

    # Share threat intelligence with MISP
    misp_sharing = misp.share_threat_intelligence(threat_indicators, incident_report)
    if misp_sharing:
        post_incident_actions.append({
            'action': 'threat_intel_sharing',
            'target': 'MISP',
            'method': 'MISP',
            'status': 'success',
            'timestamp': post_incident_time
        })
        print(f"   Threat intelligence shared with MISP community")

except Exception as e:
    print(f"   Error generating incident report: {e}")

# 2. Conduct lessons learned analysis
print(f"\n[POST-INCIDENT] Conducting lessons learned analysis...")
try:
    lessons_learned = {
        'incident_type': 'Data Exfiltration Over C2 Channel (T1041)',
        'root_cause': 'Insufficient monitoring of outbound data transfers over C2 protocols',
        'detection_effectiveness': 'High - Automated detection via Splunk and CrowdStrike',
        'response_effectiveness': 'High - Automated containment and eradication successful',
        'recovery_effectiveness': 'High - Systems restored with integrity verification',
        'gaps_identified': [
            'Need enhanced outbound traffic monitoring and analysis',
            'Consider implementing data loss prevention controls',
            'Improve detection of unusual data transfer patterns'
        ],
        'improvements_needed': [
            'Update detection rules for new exfiltration techniques',
            'Implement automated response for data exfiltration',
            'Enhance network monitoring and filtering capabilities'
        ],
        'preventive_measures': [
            'Deploy data loss prevention (DLP) solutions',
            'Implement outbound traffic monitoring and alerting',
            'Regular security assessments for data protection',
            'Enhanced user training on data handling and exfiltration risks'
        ]
    }

    # Document lessons learned in IRIS
    lessons_doc = iris.document_lessons_learned(lessons_learned)
    if lessons_doc:
        post_incident_actions.append({
            'action': 'lessons_learned',
            'target': 'IRIS',
            'method': 'IRIS',
            'status': 'success',
            'timestamp': post_incident_time
        })
        print(f"   Lessons learned documented in IRIS")

except Exception as e:
    print(f"   Error conducting lessons learned: {e}")

# 3. Implement preventive measures
print(f"\n[POST-INCIDENT] Implementing preventive measures...")
try:
    preventive_actions = []

    # Update Splunk monitoring rules
    enhanced_monitoring = splunk.implement_enhanced_monitoring('data_exfiltration_c2')
    if enhanced_monitoring:
        preventive_actions.append('Enhanced Splunk monitoring for data exfiltration')
        print(f"   Enhanced Splunk monitoring rules implemented")

    # Update CrowdStrike policies
    policy_updates = crowdstrike.update_policies('data_exfiltration_protection')
    if policy_updates:
        preventive_actions.append('Updated CrowdStrike endpoint policies')
        print(f"   CrowdStrike policies updated for data exfiltration protection")

    # Implement Shuffle automated responses
    auto_response = shuffle.implement_automated_response('data_exfiltration_c2')
    if auto_response:
        preventive_actions.append('Automated response workflows implemented')
        print(f"   Automated response workflows implemented in Shuffle")

    post_incident_actions.append({
        'action': 'preventive_measures',
        'target': preventive_actions,
        'method': 'Multiple',
        'status': 'success',
        'timestamp': post_incident_time
    })

except Exception as e:
    print(f"   Error implementing preventive measures: {e}")

# 4. Update security policies and procedures
print(f"\n[POST-INCIDENT] Updating security policies...")
try:
    policy_updates = {
        'data_exfiltration_policy': {
            'description': 'Enhanced data exfiltration monitoring and prevention',
            'changes': [
                'Added monitoring for suspicious outbound data transfers',
                'Implemented automated response for exfiltration attempts',
                'Enhanced network traffic analysis and filtering',
                'Added data loss prevention controls'
            ]
        },
        'incident_response_procedure': {
            'description': 'Updated IR procedures for data exfiltration incidents',
            'changes': [
                'Added automated containment steps for exfiltration',
                'Enhanced eradication procedures for data transfer tools',
                'Improved recovery validation for data integrity'
            ]
        }
    }

    # Update policies in Shuffle (workflow management)
    policy_update_result = shuffle.update_security_policies(policy_updates)
    if policy_update_result:
        post_incident_actions.append({
            'action': 'policy_updates',
            'target': 'Shuffle',
            'method': 'Shuffle',
            'status': 'success',
            'timestamp': post_incident_time
        })
        print(f"   Security policies updated in Shuffle")

except Exception as e:
    print(f"   Error updating security policies: {e}")

# 5. Close incident case
print(f"\n[POST-INCIDENT] Closing incident case...")
try:
    closure_summary = {
        'incident_id': incident_id,
        'closure_time': post_incident_time,
        'resolution_status': 'Resolved - Data exfiltration contained and eradicated',
        'final_status': 'Closed',
        'total_actions': len(detection_actions) + len(containment_actions) + len(eradication_actions) + len(recovery_actions) + len(post_incident_actions),
        'affected_assets': len(affected_systems),
        'threat_actor': 'Unknown - Automated response prevented further data exfiltration',
        'data_compromised': f'Data exfiltration totaling {total_data_exfiltrated:.1f} MB - contained before complete transfer',
        'business_impact': 'High - Data exfiltration prevented, but sensitive data may have been compromised'
    }

    # Close case in IRIS
    case_closure = iris.close_incident_case(incident_id, closure_summary)
    if case_closure:
        post_incident_actions.append({
            'action': 'case_closure',
            'target': f"IRIS Case {incident_id}",
            'method': 'IRIS',
            'status': 'success',
            'timestamp': post_incident_time
        })
        print(f"   Incident case closed in IRIS: {incident_id}")

    # Archive incident data
    archive_result = shuffle.archive_incident_data(incident_id)
    if archive_result:
        print(f"   Incident data archived for compliance and analysis")

except Exception as e:
    print(f"   Error closing incident case: {e}")

print(f"\n Post-incident actions complete:")
print(f"  - Incident report generated: ")
print(f"  - Lessons learned documented: ")
print(f"  - Preventive measures implemented: ")
print(f"  - Security policies updated: ")
print(f"  - Incident case closed: ")
print(f"\n Data Exfiltration Over C2 Channel Incident Response Complete")
print(f"   Incident ID: {incident_id}")
print(f"   Total Response Time: Automated (minutes)")
print(f"   Systems Protected: {len(affected_systems)}")
print(f"   Data Exfiltration Prevented: {total_data_exfiltrated:.1f} MB")
print(f"   Threat Contained: ")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
